In [2]:
import os
import sys
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cmocean
CMAP = cmocean.cm.balance
import geopandas as gpd

In [1]:
# =============================================================================
# Eddies and sections 
# =============================================================================
EDDIES = {
    "CE18": dict(
        name="CE18_20260523", eddy_type=-1,
        sections={
            # straight NE->SW crossing through the center (st. 071 -> 073 -> 074, 13 km from center)
            "crossing_0716": dict(mode="section", t0="2026-07-16T06:00", t1="2026-07-17T16:00"),
            # western flank, 54-89 km from the center (st. 044 -> 069)
            "flank_0710": dict(mode="section", t0="2026-07-10T10:00", t1="2026-07-12T17:00"),
        }),
    "AE11": dict(
        name="AE11_20260514", eddy_type=1,
        sections={
            # stations 005-048: 27 to 186 km from the center, all around the eddy
            "radial_0701_0710": dict(mode="radial", t0="2026-07-01T00:00", t1="2026-07-11T00:00",
                                     r_limit_km=220),
        }),
}

In [ ]:
SADCP_AVG = "10min"          # SADCP time averaging (~3 km at 10 kn)
USE_DETIDED = True           # use UVEL_ADCP_CORTIDE / VVEL_ADCP_CORTIDE when present
Z_SPLIT = 150.0              # m: OS150 above, OS38 below
ZMAX_SADCP = 1000.0
ZMAX_LADCP = 1500.0
DS_KM = 3.0                  # bin size along a section (SADCP)
R_BIN_KM = 10.0              # radius bin (radial mode, SADCP)
LADCP_MAX_GAP_KM = 60.0      # do not interpolate between LADCP stations further apart
FIT_CENTER = True            # radial mode: refine the TOEddies center from SADCP
FIT_LAYER = (30.0, 200.0)    # m: layer used to fit the center
FIT_SEARCH_KM = 60.0         # search +- this offset around the TOEddies center
CENTER_SMOOTH_DAYS = 3       # running mean on the daily TOEddies centers (they move on a 1/8° grid)
PROFILE_DEPTHS = [50, 100, 200, 400, 700]   # radial mode: v_theta(r) at these depths
R_KM = 6371.0
crs = ccrs.PlateCarree()
